# Predicting the Unpredictable — Introductory Demo


This notebook covers only the preliminary phases:
1. **Downloading** traffic data (Bologna municipality loop detectors) and weather data (Open-Meteo)
2. **Merging** the two datasets on timestamp
3. **Basic visualisations** to explore the data

This is a quick demo: we work with **few detectors** and a **short period** to avoid overloading the network and computation.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

# Demo parameters: short period and few detectors
ANNO = 2022
DATA_INIZIO = '2023-09-01'
DATA_FINE   = '2025-09-30'
N_SPIRE_DEMO = 100    # limit to a few detectors for speed

# Bologna coordinates for weather
LAT, LON = 44.49, 11.34

print(f'Demo period: {DATA_INIZIO} → {DATA_FINE}')
print(f'Number of selected detectors: {N_SPIRE_DEMO}')

## 1. Downloading traffic data (Bologna municipality loop detectors)

Data are published on the Bologna Open Data portal. We use the Opendatasoft API with a `where` filter on date to download only the period of interest.

**Raw data schema**: each row = one loop detector on a specific day, with 24 hourly columns (`00:00-01:00`, `01:00-02:00`, ..., `23:00-24:00`). We will **melt** the dataset into *long* format (one row per hourly timestamp).

In [ ]:
def scarica_spire(anno: int, data_inizio: str, data_fine: str) -> pd.DataFrame:
    """Download hourly flow data for Bologna loop detectors over a date range.

    We use the /exports/json endpoint which has no page limits (the standard
    Opendatasoft /records endpoint saturates at offset=10000).
    """
    dataset_id = f'rilevazione-flusso-veicoli-tramite-spire-anno-{anno}'
    url = f'https://opendata.comune.bologna.it/api/explore/v2.1/catalog/datasets/{dataset_id}/exports/json'
    params = {
        "where": f"data >= '{data_inizio}' AND data <= '{data_fine}'",
    }
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    return pd.DataFrame(r.json())


def scarica_spire_periodo(data_inizio: str,
                          data_fine: str) -> pd.DataFrame:

    anno_inizio = pd.to_datetime(data_inizio).year
    anno_fine   = pd.to_datetime(data_fine).year

    dfs = []

    for anno in range(anno_inizio, anno_fine + 1):

        print(f"Downloading year {anno}...")

        df = scarica_spire(
            anno=anno,
            data_inizio=data_inizio,
            data_fine=data_fine
        )

        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

# Local cache: download once only
file_spire = DATA_DIR / f'spire_{DATA_INIZIO}_{DATA_FINE}.parquet'
if file_spire.exists():
    df_spire_raw = pd.read_parquet(file_spire)
    print(f'Loaded from cache: {file_spire}')
else:
    df_spire_raw = scarica_spire(ANNO, DATA_INIZIO, DATA_FINE)
    df_spire_raw.to_parquet(file_spire)
    print(f'Downloaded and saved: {file_spire}')

df_spire_raw = scarica_spire_periodo(
    DATA_INIZIO,
    DATA_FINE
)
df_spire_raw.to_parquet(file_spire)

print(f'Raw shape: {df_spire_raw.shape}')
df_spire_raw.head(3)

In [ ]:
import plotly.express as px
import plotly.io as pio
import os
os.environ["BROWSER"] = "chrome" 
pio.renderers.default = "browser"

def map_coils(df, chiave='chiave'):
    fig = px.scatter_mapbox(
        df[[chiave, 'latitudine', 'longitudine']].drop_duplicates(subset=chiave),
        lat="latitudine",
        lon="longitudine",
        hover_name=chiave,
        zoom=11,
        height=600
    )
    fig.update_traces(marker=dict(color="red"))
    fig.update_layout(
        mapbox_style="open-street-map",  # no token required
        margin={"r":0, "t":0, "l":0, "b":0}
    )

    fig.write_html("mappa.html")

def sottocampiona_griglia(
    df: pd.DataFrame,
    lat_col: str = "latitudine",
    lon_col: str = "longitudine",
    id_col: str = "chiave",
    n_lat: int = 10,
    n_lon: int = 10,
    mode: str = "centroid"  # "centroid" or "random"
) -> pd.DataFrame:
    """
    Subsample sensors using a geographic grid.

    Parameters
    ----------
    df : DataFrame
        Must contain latitudine and longitudine columns
    n_lat : int
        number of latitude cells
    n_lon : int
        number of longitude cells
    mode : str
        - "centroid": pick the sensor closest to the cell centre
        - "random": pick a random sensor per cell
    """

    df = df[[id_col, 'latitudine', 'longitudine']].drop_duplicates(subset=id_col).copy()

    lat_min, lat_max = df[lat_col].min(), df[lat_col].max()
    lon_min, lon_max = df[lon_col].min(), df[lon_col].max()

    # cell dimensions
    lat_bins = np.linspace(lat_min, lat_max, n_lat + 1)
    lon_bins = np.linspace(lon_min, lon_max, n_lon + 1)

    # assign cell
    df["lat_bin"] = np.digitize(df[lat_col], lat_bins) - 1
    df["lon_bin"] = np.digitize(df[lon_col], lon_bins) - 1

    selected = []

    for (_, group) in df.groupby(["lat_bin", "lon_bin"]):

        if len(group) == 0:
            continue

        if mode == "random":
            selected.append(group.sample(1))

        elif mode == "centroid":
            # cell centre
            lat_center = group[lat_col].mean()
            lon_center = group[lon_col].mean()

            # distance from centre
            dist = (group[lat_col] - lat_center)**2 + (group[lon_col] - lon_center)**2

            selected.append(group.loc[[dist.idxmin()]])

        else:
            raise ValueError("mode must be 'centroid' or 'random'")

    return pd.concat(selected).reset_index(drop=True)

In [4]:
df_usamp = sottocampiona_griglia(df_spire_raw, n_lat=6, n_lon=5,)
map_coils(df_usamp, chiave="chiave")

/tmp/ipykernel_26938/2617415248.py:18: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


In [ ]:
# Take a look at available columns
print('Columns:', list(df_spire_raw.columns))

In [ ]:
# count unique detectors
n_spire_uniche = df_spire_raw['codice_spira'].nunique()
print(f'Number of unique detectors: {n_spire_uniche}')

In [7]:
df_spire_sel = df_spire_raw[df_spire_raw['id_uni'].isin(df_usamp['chiave'])]

### Selecting a few detectors and melting to long format

We select only `N_SPIRE_DEMO` detectors (those with the most readings in the period) and reshape the data to *long* format: one row = `(detector, hourly timestamp, count)`.

In [ ]:
# 1) Select top detectors in the period.
#    We use 'chiave' (sensor identifier) rather than id_uni alone because
#    in some cases the same id_uni covers multiple distinct sensors (e.g. two
#    directions on the same station): selecting by id_uni alone produces
#    duplicate records in subsequent merges.
top_spire = (
    df_spire_sel.groupby(['id_uni', 'chiave']).size()
      .sort_values(ascending=False)
      .index.tolist()  # list of (id_uni, chiave) tuples
)
print('Selected detectors (id_uni, chiave):', top_spire)

mask = df_spire_sel.set_index(['id_uni', 'chiave']).index.isin(top_spire)
df_sel = df_spire_sel[mask].copy()

# 2) Hourly columns in HH_00_HH_00 format
import re
pat = re.compile(r'^\d{2}_\d{2}_\d{2}_\d{2}$')
colonne_orarie = [c for c in df_sel.columns if pat.match(c)]
print(f'Hourly columns found: {len(colonne_orarie)}')

# 3) Melt to long format
id_vars = ['data', 'id_uni', 'chiave']
for opt in ['nome_via', 'direzione', 'longitudine', 'latitudine']:
    if opt in df_sel.columns:
        id_vars.append(opt)

df_long = df_sel.melt(
    id_vars=id_vars,
    value_vars=colonne_orarie,
    var_name='fascia_oraria',
    value_name='conteggio_veicoli',
)

# 4) Hourly timestamp
df_long['ora'] = df_long['fascia_oraria'].str[:2].astype(int)
df_long['timestamp'] = pd.to_datetime(df_long['data']) + pd.to_timedelta(df_long['ora'], unit='h')
df_long['conteggio_veicoli'] = pd.to_numeric(df_long['conteggio_veicoli'], errors='coerce')

df_long = df_long.sort_values(['chiave', 'timestamp']).reset_index(drop=True)
print('')
print(f'Long dataset shape: {df_long.shape}')
df_long[['timestamp', 'id_uni', 'chiave', 'nome_via', 'conteggio_veicoli']].head()

## 2. Downloading weather data (Open-Meteo)

Open-Meteo provides a free historical endpoint without an API key. We download hourly temperature, precipitation and wind speed for Bologna over the same period.

In [ ]:
def scarica_meteo(lat: float, lon: float, data_inizio: str, data_fine: str) -> pd.DataFrame:
    """Download hourly weather data from Open-Meteo for a location and period."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': data_inizio,
        'end_date': data_fine,
        'hourly': 'temperature_2m,precipitation,rain,wind_speed_10m,weather_code',
        'timezone': 'Europe/Rome',
    }
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    h = r.json()['hourly']
    df = pd.DataFrame(h)
    df['timestamp'] = pd.to_datetime(df['time'])
    return df.drop(columns=['time'])

file_meteo = DATA_DIR / f'meteo_{DATA_INIZIO}_{DATA_FINE}.parquet'
if file_meteo.exists():
    df_meteo = pd.read_parquet(file_meteo)
    print(f'Loaded from cache: {file_meteo}')
else:
    df_meteo = scarica_meteo(LAT, LON, DATA_INIZIO, DATA_FINE)
    df_meteo.to_parquet(file_meteo)
    print(f'Downloaded and saved: {file_meteo}')

print(f'Weather shape: {df_meteo.shape}')
df_meteo.head()

### `weather_code` mapping table (WMO)

Open-Meteo returns a numeric WMO code. We convert it to human-readable labels: useful both for plots and for anyone using the data downstream.

Reference: [WMO Weather interpretation codes](https://open-meteo.com/en/docs/historical-weather-api) (section *WMO Weather interpretation codes*).

In [ ]:
# WMO code → human-readable label
WMO_MAP = {
    0:  'Clear sky',
    1:  'Mainly clear',
    2:  'Partly cloudy',
    3:  'Overcast',
    45: 'Fog',
    48: 'Depositing rime fog',
    51: 'Light drizzle',
    53: 'Moderate drizzle',
    55: 'Dense drizzle',
    56: 'Light freezing drizzle',
    57: 'Heavy freezing drizzle',
    61: 'Slight rain',
    63: 'Moderate rain',
    65: 'Heavy rain',
    66: 'Light freezing rain',
    67: 'Heavy freezing rain',
    71: 'Slight snowfall',
    73: 'Moderate snowfall',
    75: 'Heavy snowfall',
    77: 'Snow grains',
    80: 'Slight rain showers',
    81: 'Moderate rain showers',
    82: 'Violent rain showers',
    85: 'Slight snow showers',
    86: 'Heavy snow showers',
    95: 'Thunderstorm',
    96: 'Thunderstorm with slight hail',
    99: 'Thunderstorm with heavy hail',
}

df_meteo['tempo'] = df_meteo['weather_code'].map(WMO_MAP).fillna('Unknown')
df_meteo['tempo'].value_counts()

## 3. Merging the two datasets

We join traffic and weather on `timestamp`. Weather is city-wide, so it is replicated across all detectors (left join on traffic).

In [ ]:
df = df_long.merge(df_meteo, on='timestamp', how='left')

# Useful temporal columns for plots
df['giorno_settimana'] = df['timestamp'].dt.day_name()
df['ora_del_giorno']   = df['timestamp'].dt.hour
df['weekend']          = df['timestamp'].dt.dayofweek >= 5

print(f'Final dataset shape: {df.shape}')
print(f'Period: {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'Detectors (distinct keys): {df["chiave"].nunique()}')
df.head()

## 4. Basic Visualisations

Four introductory plots to get a sense of the data structure.

### 4.1 Flow time series, one detector

In [ ]:
id_uni_demo, chiave_demo = top_spire[5]
df_one = (
    df[(df['id_uni'] == id_uni_demo) & (df['chiave'] == chiave_demo)]
      .sort_values('timestamp')
)
via = df_one['nome_via'].iloc[0] if 'nome_via' in df_one.columns else id_uni_demo

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_one['timestamp'], df_one['conteggio_veicoli'], linewidth=0.8)
ax.set_title(f'Hourly flow — id_uni {id_uni_demo} / chiave {chiave_demo} ({via})')
ax.set_xlabel('Time')
ax.set_ylabel('Vehicles/hour')
plt.tight_layout()
plt.show()

### 4.2 Average hourly profile by day of week

The classic weekday pattern (double peak at rush hours) vs. weekend.

In [ ]:
ordine_giorni = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

profilo = (
    df.groupby(['giorno_settimana', 'ora_del_giorno'])['conteggio_veicoli']
      .mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for g in ordine_giorni:
    sub = profilo[profilo['giorno_settimana'] == g]
    ax.plot(sub['ora_del_giorno'], sub['conteggio_veicoli'], marker='o', label=g)
ax.set_title('Average hourly profile by day of week')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Vehicles/hour (average across all detectors)')
ax.set_xticks(range(0, 24))
ax.legend(ncol=4, fontsize=9)
plt.tight_layout()
plt.show()

### 4.3 Traffic heatmap (hour × day)

An overview: how much traffic passes, on average, in each `(day × hour)` cell.

In [ ]:
pivot = (
    df.groupby(['giorno_settimana', 'ora_del_giorno'])['conteggio_veicoli']
      .mean()
      .unstack()
      .reindex(ordine_giorni)
)

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, cmap='viridis', cbar_kws={'label': 'Vehicles/hour'}, ax=ax)
ax.set_title('Average traffic heatmap — day of week × hour')
ax.set_xlabel('Hour of day')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 4.4 Traffic vs. rain

A first look at the weather effect: does flow change when it rains?

In [ ]:
df['pioggia'] = df['precipitation'].fillna(0) > 0.1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# (a) Boxplot: flow distribution with/without rain
sns.boxplot(data=df, x='pioggia', y='conteggio_veicoli', ax=axes[0])
axes[0].set_title('Flow distribution: rain vs. dry')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Dry', 'Rain'])
axes[0].set_xlabel('')
axes[0].set_ylabel('Vehicles/hour')

# (b) Scatter: temperature vs. aggregated hourly flow
agg = df.groupby('timestamp').agg(
    flusso=('conteggio_veicoli', 'mean'),
    temperatura=('temperature_2m', 'first'),
    pioggia=('precipitation', 'first'),
).reset_index()
sns.scatterplot(data=agg, x='temperatura', y='flusso', hue='pioggia',
                palette='viridis', s=20, ax=axes[1])
axes[1].set_title('Average hourly flow vs. temperature')
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Vehicles/hour (detector average)')

plt.tight_layout()
plt.show()

## 5. Detector Accuracy Dataset

The project description is clear: this dataset is *not a technical detail, it is the heart of the project*. It lets us distinguish a broken detector (accuracy 0%) from a genuinely empty road (accuracy 100% with count 0).

Schema similar to the flow dataset but with some differences:
- renamed keys (`data_2`, `codice_spira_2`)
- hourly columns in `HH_00_HH` format (3 segments instead of 4)
- values as strings `'85%'` to be converted to numeric
- the join with the main dataset uses the `chiave` field (present in both)

In [ ]:
def scarica_accuratezza(data_inizio: str, data_fine: str,
                        cache_dir: str = 'data/cache') -> pd.DataFrame:
    os.makedirs(cache_dir, exist_ok=True)
    cache_path = os.path.join(cache_dir, f'accuratezza_{data_inizio}_{data_fine}.parquet')

    if os.path.exists(cache_path):
        print(f'[cache] Loading from disk: {cache_path}')
        return pd.read_parquet(cache_path)
    anno_inizio = pd.to_datetime(data_inizio).year
    anno_fine   = pd.to_datetime(data_fine).year

    dfs = []
    for anno in range(anno_inizio, anno_fine + 1):
        dataset_id = f'accuratezza-spire-anno-{anno}'
        url = (
            f'https://opendata.comune.bologna.it/api/explore/v2.1/'
            f'catalog/datasets/{dataset_id}/exports/json'
        )
        params = {'where': f"data_2 >= '{data_inizio}' AND data_2 <= '{data_fine}'"}

        print(f'[download] Downloading accuracy {anno}...')
        r = requests.get(url, params=params, timeout=120)
        r.raise_for_status()

        df = pd.DataFrame(r.json())
        if not df.empty:
            df['anno_dataset'] = anno
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()
    
    df_finale = pd.concat(dfs, ignore_index=True)

    df_finale.to_parquet(cache_path)

    return df_finale

file_acc = DATA_DIR / f'accuratezza_{DATA_INIZIO}_{DATA_FINE}.parquet'

df_acc_raw = scarica_accuratezza(
    DATA_INIZIO,
    DATA_FINE
)
df_acc_raw.to_parquet(file_acc)
print(f'Downloaded and saved: {file_acc}')

print(f'Raw accuracy shape: {df_acc_raw.shape}')
df_acc_raw.head(2)

In [ ]:
# Detector 'keys' are already in the dataframe
chiavi_demo = df['chiave'].dropna().unique().tolist()
print(f'Accuracy keys to filter: {chiavi_demo}')

df_acc_sel = df_acc_raw[df_acc_raw['chiave'].isin(chiavi_demo)].copy()

# Hourly columns here are HH_00_HH (3 segments)
pat_acc = re.compile(r'^\d{2}_\d{2}_\d{2}$')
colonne_orarie_acc = [c for c in df_acc_sel.columns if pat_acc.match(c)]
print(f'Accuracy hourly columns: {len(colonne_orarie_acc)}')

df_acc_long = df_acc_sel.melt(
    id_vars=['data_2', 'chiave'],
    value_vars=colonne_orarie_acc,
    var_name='fascia_oraria_acc',
    value_name='accuratezza_str',
)

df_acc_long['ora'] = df_acc_long['fascia_oraria_acc'].str[:2].astype(int)
df_acc_long['timestamp'] = (
    pd.to_datetime(df_acc_long['data_2']) + pd.to_timedelta(df_acc_long['ora'], unit='h')
)
df_acc_long['accuratezza'] = (
    df_acc_long['accuratezza_str'].astype(str).str.rstrip('%').replace('', np.nan).astype(float)
)
# Negative values (e.g. -1) are a sentinel for 'missing data': set to NaN
df_acc_long.loc[df_acc_long['accuratezza'] < 0, 'accuratezza'] = np.nan

df_acc_long = df_acc_long[['chiave', 'timestamp', 'accuratezza']]
print(f'Accuracy long shape: {df_acc_long.shape}')
df_acc_long.head()

In [ ]:
# 'chiave' is already in df (set during initial selection), merge is direct
df = df.merge(df_acc_long, on=['chiave', 'timestamp'], how='left')

print(f'Shape after accuracy merge: {df.shape}')
print(
    f"Accuracy: mean={df['accuratezza'].mean():.1f}%  "
    f"min={df['accuratezza'].min():.0f}%  NaN={df['accuratezza'].isna().sum()}"
)
df[['timestamp', 'id_uni', 'chiave', 'conteggio_veicoli', 'accuratezza']].head()

## 6. Italian Public Holidays Calendar

We use the `holidays` package, which knows Italian national holidays. We also add two derived columns useful for problem framing: `tipo_giorno` (weekday / weekend / holiday) and `festa_locale` for San Petronio (4 October), which `holidays` does not include by default.

In [ ]:
import holidays

# National and local holidays
anni = sorted(df['timestamp'].dt.year.unique().tolist())
festivita_it = holidays.country_holidays('IT', years=anni)

festivita_locali = {f'{a}-10-04': 'San Petronio' for a in anni}

# High-impact traffic events
# Format: (start_date, end_date, name, type, impact)
# For single-day matches/events: start_date == end_date
_eventi_raw = [
    # 2023
    ('2023-09-03', '2023-09-03', 'Bologna vs Cagliari',          'calcio_serieA', 'alto'),
    ('2023-09-07', '2023-09-10', 'SANA 2023',                    'fiera',         'alto'),
    ('2023-09-24', '2023-09-24', 'Bologna vs Napoli',            'calcio_serieA', 'molto_alto'),
    ('2023-09-25', '2023-09-29', 'CERSAIE 2023',                 'fiera',         'molto_alto'),
    ('2023-10-22', '2023-10-22', 'Bologna vs Frosinone',         'calcio_serieA', 'alto'),
    ('2023-10-26', '2023-10-29', "Auto e Moto d'Epoca 2023",     'fiera',         'molto_alto'),
    ('2023-11-05', '2023-11-05', 'Bologna vs Lazio',             'calcio_serieA', 'alto'),
    ('2023-11-26', '2023-11-26', 'Bologna vs Torino',            'calcio_serieA', 'alto'),
    ('2023-12-17', '2023-12-17', 'Bologna vs Roma',              'calcio_serieA', 'alto'),
    ('2023-12-23', '2023-12-23', 'Bologna vs Atalanta',          'calcio_serieA', 'alto'),
    # 2024
    ('2024-03-21', '2024-03-24', 'Cosmoprof 2024',               'fiera',         'molto_alto'),
    ('2024-08-18', '2024-08-18', 'Bologna vs Udinese',           'calcio_serieA', 'alto'),
    ('2024-09-05', '2024-09-08', 'SANA 2024',                    'fiera',         'alto'),
    ('2024-09-18', '2024-09-18', 'Bologna vs Shakhtar UCL',      'calcio_UCL',    'molto_alto'),
    ('2024-09-23', '2024-09-27', 'CERSAIE 2024',                 'fiera',         'molto_alto'),
    ('2024-09-28', '2024-09-28', 'Bologna vs Atalanta',          'calcio_serieA', 'alto'),
    ('2024-10-24', '2024-10-27', "Auto e Moto d'Epoca 2024",     'fiera',         'molto_alto'),
    ('2024-10-26', '2024-10-26', 'Bologna vs Milan',             'calcio_serieA', 'molto_alto'),
    ('2024-11-02', '2024-11-02', 'Bologna vs Lecce',             'calcio_serieA', 'alto'),
    ('2024-11-05', '2024-11-05', 'Bologna vs Monaco UCL',        'calcio_UCL',    'alto'),
    ('2024-11-06', '2024-11-10', 'EIMA International 2024',      'fiera',         'alto'),
    ('2024-11-27', '2024-11-27', 'Bologna vs Lille UCL',         'calcio_UCL',    'alto'),
    # 2025
    ('2025-01-21', '2025-01-21', 'Bologna vs Borussia Dortmund', 'calcio_UCL',    'molto_alto'),
    ('2025-04-20', '2025-04-20', 'Bologna vs Inter',             'calcio_serieA', 'molto_alto'),
    ('2025-09-20', '2025-09-20', 'Bologna vs Genoa',             'calcio_serieA', 'alto'),
    ('2025-10-02', '2025-10-02', 'Bologna vs Friburgo UEL',      'calcio_UEL',    'alto'),
]

# Expand multi-day ranges into daily dicts
# If a day has multiple events (e.g. trade fair + match), concatenate with " | "
_eventi_nome:    dict[str, str] = {}
_eventi_tipo:    dict[str, str] = {}
_eventi_impatto: dict[str, str] = {}
_SCORE = {'molto_alto': 2, 'alto': 1}

for data_ini, data_fin, nome, tipo, impatto in _eventi_raw:
    for d in pd.date_range(data_ini, data_fin).date:
        key = str(d)
        if key in _eventi_nome:
            _eventi_nome[key]    += f' | {nome}'
            # type: keep most specific (UCL > serieA > fiera)
            _eventi_tipo[key]     = _eventi_tipo[key] if _eventi_tipo[key] != 'fiera' else tipo
            # impact: take maximum
            if _SCORE[impatto] > _SCORE[_eventi_impatto[key]]:
                _eventi_impatto[key] = impatto
        else:
            _eventi_nome[key]    = nome
            _eventi_tipo[key]    = tipo
            _eventi_impatto[key] = impatto

# Feature engineering: holidays, events, day categories
df['data_giorno']       = df['timestamp'].dt.date
df['data_str']          = df['timestamp'].dt.strftime('%Y-%m-%d')

df['festivo_nazionale'] = df['data_giorno'].apply(lambda d: d in festivita_it)
df['nome_festivita']    = df['data_giorno'].apply(lambda d: festivita_it.get(d))
df['festa_locale']      = df['data_str'].map(festivita_locali).notna()

df['evento_traffico']   = df['data_str'].map(_eventi_nome)
df['tipo_evento']       = df['data_str'].map(_eventi_tipo)
df['impatto_evento']    = df['data_str'].map(_eventi_impatto)
df['score_evento']      = df['impatto_evento'].map(_SCORE).fillna(0).astype(int)
df['eventi_sovrapposti'] = df['data_str'].map(
    lambda s: _eventi_nome.get(s, '').count(' | ')  # overlap count = number of " | " separators
)

def categoria_giorno(row):
    if row['festivo_nazionale'] or row['festa_locale']:
        return 'festivo'
    if row['weekend']:
        return 'weekend'
    return 'feriale'

df['tipo_giorno'] = df.apply(categoria_giorno, axis=1)

# Diagnostics
print('Day type distribution (hours):')
print(df['tipo_giorno'].value_counts())

print('\nHolidays found in the period:')
fest = df.loc[df['festivo_nazionale'], ['data_giorno', 'nome_festivita']].drop_duplicates()
print(fest.to_string(index=False) if len(fest) else '(none in the selected period)')

print('\nHigh-impact events found in the period:')
ev = (
    df.loc[df['evento_traffico'].notna(),
           ['data_giorno', 'evento_traffico', 'tipo_evento', 'impatto_evento', 'score_evento']]
    .drop_duplicates('data_giorno')
    .sort_values('data_giorno')
)
print(ev.to_string(index=False) if len(ev) else '(none in the selected period)')

print('\nDays with overlapping events (trade fair + match):')
sov = (
    df.loc[df['eventi_sovrapposti'] > 0,
           ['data_giorno', 'evento_traffico', 'score_evento']]
    .drop_duplicates('data_giorno')
    .sort_values('data_giorno')
)
print(sov.to_string(index=False) if len(sov) else '(none)')

## 7. Visualisations with the enriched data

### 7.1 Detector accuracy heatmap

An overview of how reliable the measurements are hour-by-hour in the period. Dark patches are where the sensor performed poorly: those are exactly the spots where the anomaly detection system risks generating false positives if data quality is not handled.

In [ ]:
acc_pivot = (
    df.assign(giorno=df['timestamp'].dt.date)
      .groupby(['chiave', 'giorno'])['accuratezza']
      .mean()
      .unstack()
)

fig, ax = plt.subplots(figsize=(14, max(2, len(acc_pivot) * 0.5)))
sns.heatmap(acc_pivot, cmap='RdYlGn', vmin=0, vmax=100,
            cbar_kws={'label': 'Average daily accuracy (%)'}, ax=ax)
ax.set_title('Coils accuracy - key per day')
ax.set_xlabel('Day')
ax.set_ylabel('Sensor key')
plt.tight_layout()
plt.show()

### 7.2 Hourly profile: weekdays vs weekends vs holidays

Expected: holidays and weekends should flatten the double-peak rush-hour pattern. If that is not the case for a particular detector, it is already an interesting signal for problem framing.

In [ ]:
profilo_g = (
    df.groupby(['tipo_giorno', 'ora_del_giorno'])['conteggio_veicoli']
      .mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for tipo, marker in zip(['feriale', 'weekend', 'festivo'], ['o', 's', '^']):
    sub = profilo_g[profilo_g['tipo_giorno'] == tipo]
    if not sub.empty:
        ax.plot(sub['ora_del_giorno'], sub['conteggio_veicoli'],
                marker=marker, label=tipo, linewidth=2)
ax.set_title('Average hourly profile by day type')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Vehicles/hour (mean)')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

In [24]:
df.to_parquet(f'../data/processed/dataset_finale_{DATA_INIZIO}_{DATA_FINE}.parquet')

### 7.3 Weather condition → vehicle count

Which weather type coincides on average with more or less traffic? Note: some categories may have very few hours — interpret with caution.

In [ ]:
conteggio_per_tempo = (
    df.groupby('tempo')['conteggio_veicoli']
      .agg(['mean', 'count'])
      .sort_values('mean', ascending=True)
)
print(conteggio_per_tempo)

fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(conteggio_per_tempo))))
ax.barh(conteggio_per_tempo.index, conteggio_per_tempo['mean'], color='steelblue')
ax.set_xlabel('Vehicles/hour (mean)')
ax.set_title('Average hourly flow by weather condition')
plt.tight_layout()
plt.show()